[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/Quantlets/Ch_08/SFM_ch8_long_term_vix/SFM_ch8_long_term_vix.ipynb)

# SFM_ch8_long_term_vix

Long-run behaviour of volatility: the volatility of each calendar year for the S&P 500, DAX, BET and Bitcoin; the AR(1) of the log monthly realised volatility of the S&P 500, its half-life and skewness; the VIX (implied volatility of the S&P 500 for the next 30 days) against the realised volatility of the next 21 trading days, the share of days with the VIX above it and Mincer-Zarnowitz regressions.

*Statistics of Financial Markets (SFM), Chapter 8: volatility estimators and volatility clustering. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/SFM/Chapter_8'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import sfm_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/SFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the SFM repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years

## Analysis

In [ ]:
NAME = {'sp500': 'S&P 500', 'dax': 'DAX', 'bet': 'BET', 'btc': 'Bitcoin', 'tlv': 'Banca Transilvania', 'snp': 'OMV Petrom', 'brd': 'BRD', 'tgn': 'Transgaz', 'vix': 'VIX'}

START = {'sp500': '1990-01-01', 'dax': '1990-01-01', 'bet': '1997-01-01', 'btc': None, 'tlv': '2010-01-01', 'snp': '2010-01-01', 'brd': '2010-01-01', 'tgn': '2010-01-01'}

OHLC_START = {'sp500': '2008-01-01', 'dax': '2006-01-01', 'btc': None, 'tlv': '2010-01-01', 'snp': '2010-01-01', 'brd': '2010-01-01', 'tgn': '2010-01-01'}

BAD_DAYS = {'tlv': ['2016-05-30', '2016-05-31']}

ASSETS = ['sp500', 'dax', 'bet', 'btc']

STOCKS = ['tlv', 'snp']

BVB = ['tlv', 'snp', 'brd', 'tgn']

OHLC_ASSETS = ['sp500', 'dax', 'btc', 'tlv', 'snp']

COLORS = {'sp500': '#1A3A6E', 'dax': '#17A2B8', 'bet': '#CD0000', 'btc': '#B5853F', 'snp': '#2E7D32', 'brd': '#8E44AD', 'tlv': '#E67E22', 'tgn': '#DC3545'}

EST_COLORS = {'cc': '#1A3A6E', 'park': '#CD0000', 'gk': '#2E7D32', 'rs': '#E67E22', 'yz': '#8E44AD'}

EST_LABEL = {'cc': 'Close-to-close', 'park': 'Parkinson', 'gk': 'Garman-Klass', 'rs': 'Rogers-Satchell', 'yz': 'Yang-Zhang'}

WINDOWS = (21, 63, 252)

LAMBDAS = (0.94, 0.97)

LB_LAGS = 10

ARCH_LAGS = 5

OOS_START = '2010-01-01'

SEED = 2026

GHOST = ('2019-10-01', '2021-03-31')

def returns(k, start=None):
    """Daily log returns in % on the series' own calendar; known data errors removed."""
    r = log_returns(k, start or START.get(k))
    if k in BAD_DAYS:
        r = r.drop(pd.to_datetime(BAD_DAYS[k]), errors='ignore')
    return r


def ann(r):
    """Annualisation factor: the actual number of observations per year (about 252 for stocks, 365 for Bitcoin)."""
    return periods_per_year(r)


def annual_vol(names=ASSETS):
    """Volatility of each calendar year (annualised with the actual frequency of the series, %)."""
    out = {}
    for k in names:
        r = returns(k)
        a = ann(r)
        g = r.groupby(r.index.year)
        v = g.std() * np.sqrt(a)
        v = v[g.size() >= 100]
        out[k] = v
    return out


def monthly_rv(k='sp500'):
    """Monthly realised volatility sqrt(a x mean of squared daily returns in the month), annualised %, and the AR(1)
    of its logarithm: log RV_m = c + phi log RV_{m-1} + e_m; half-life ln 0.5 / ln phi months."""
    r = returns(k)
    a = ann(r)
    m = np.sqrt((r ** 2).groupby(r.index.to_period('M')).mean() * a)
    lm = np.log(m)
    y, x = lm.values[1:], lm.values[:-1]
    X = np.column_stack([np.ones(len(x)), x])
    b, *_ = np.linalg.lstsq(X, y, rcond=None)
    u = y - X @ b
    se = np.sqrt(np.sum(u ** 2) / (len(y) - 2) * np.linalg.inv(X.T @ X)[1, 1])
    return m, {'phi': float(b[1]), 'c': float(b[0]), 'se': float(se), 'half_life': float(np.log(0.5) / np.log(b[1])),
               'skew_rv': float(stats.skew(m)), 'skew_log': float(stats.skew(lm)), 'mean_rv': float(m.mean()),
               'median_rv': float(m.median()), 'n': len(m), 'long_run': float(np.exp(b[0] / (1 - b[1])))}


def fig_long_term(names=ASSETS, save=True):
    """Annual volatility of four markets since 1990 (left); log monthly realised volatility of the S&P 500 against its
    value in the previous month (right): high volatility persists but reverts to its mean."""
    av = annual_vol(names)
    m, ar = monthly_rv('sp500')
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.0), gridspec_kw={'width_ratios': [1.6, 1]})
    for k in names:
        axes[0].plot(av[k].index, av[k].values, color=COLORS[k], marker='o', ms=3, lw=1.3, label=NAME[k])
    axes[0].set_ylabel('Annual volatility (%)')
    axes[0].set_xlabel('Year')
    lm = np.log(m.values)
    axes[1].scatter(lm[:-1], lm[1:], s=8, color='#1A3A6E', alpha=0.6, label='S&P 500, month m-1 against m')
    g = np.linspace(lm.min(), lm.max(), 50)
    axes[1].plot(g, ar['c'] + ar['phi'] * g, color='#CD0000', lw=1.6, label=f'AR(1) line, phi = {ar["phi"]:.2f}')
    axes[1].plot(g, g, color='black', lw=0.7, ls='--', label='45-degree line')
    axes[1].set_xlabel('log RV, previous month')
    axes[1].set_ylabel('log RV, this month')
    st.fig_legend_bottom(fig, ncol=4, y=0.0)
    fig.tight_layout(rect=(0, 0.1, 1, 1))
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch8_long_term')
    out = {'ar': ar}
    for k in names:
        v = av[k]
        out[k] = {'max': float(v.max()), 'max_year': int(v.idxmax()), 'min': float(v.min()), 'min_year': int(v.idxmin()),
                  'median': float(v.median()), 'y2008': float(v.get(2008, np.nan)), 'y2020': float(v.get(2020, np.nan)),
                  'last': float(v.iloc[-1]), 'last_year': int(v.index[-1])}
    return out


def vix_data(h=21):
    """VIX close and the realised volatility of the S&P 500 over the next h trading days,
    sqrt(a/h x sum_{j=1..h} r_{t+j}^2) (annualised %), and over the past h days, on the S&P 500 calendar."""
    r = returns('sp500')
    a = ann(r)
    v = load_close('vix', START['sp500']).reindex(r.index).ffill()
    r2 = r ** 2
    fwd = np.sqrt(r2[::-1].rolling(h).sum()[::-1].shift(-1) * a / h)
    past = np.sqrt(r2.rolling(h).mean() * a)
    return pd.DataFrame({'vix': v, 'rv_fwd': fwd, 'rv_past': past, 'r': r}).dropna(subset=['vix'])


def ols(y, x):
    """OLS of y on a constant and x: intercept, slope, R^2."""
    X = np.column_stack([np.ones(len(x)), x])
    b, *_ = np.linalg.lstsq(X, y, rcond=None)
    u = y - X @ b
    return {'a': float(b[0]), 'b': float(b[1]), 'r2': float(1 - np.sum(u ** 2) / np.sum((y - np.mean(y)) ** 2))}


def fig_vix(save=True):
    """VIX (implied volatility of the S&P 500 for the next 30 calendar days) against the realised volatility of the
    next 21 trading days."""
    d = vix_data()
    z = d.dropna()
    fig, ax = plt.subplots(figsize=(10, 4.0))
    ax.plot(d.index, d['vix'], color='#CD0000', lw=0.9, label='VIX (implied volatility, %)')
    ax.plot(z.index, z['rv_fwd'], color='#1A3A6E', lw=0.9, label='Realised volatility of the next 21 days (%)')
    ax.set_ylabel('Annualised volatility (%)')
    ax.set_xlabel('Date')
    st.legend_outside_bottom(ax, ncol=2, y=-0.18)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch8_vix')
    dv = np.log(d['vix']).diff()
    m = pd.concat([dv, d['r']], axis=1).dropna()
    return {'n': len(z), 'first': d.index[0], 'mean_vix': float(z['vix'].mean()), 'mean_rv': float(z['rv_fwd'].mean()),
            'share_above': float(np.mean(z['vix'] > z['rv_fwd'])), 'gap': float((z['vix'] - z['rv_fwd']).mean()),
            'vrp_var': float((z['vix'] ** 2 - z['rv_fwd'] ** 2).mean()), 'corr': float(np.corrcoef(z['vix'], z['rv_fwd'])[0, 1]),
            'mz_vix': ols(z['rv_fwd'].values, z['vix'].values), 'mz_past': ols(z['rv_fwd'].values, z['rv_past'].values),
            'corr_dvix_r': float(np.corrcoef(m.iloc[:, 0], m.iloc[:, 1])[0, 1]), 'max_vix': float(d['vix'].max()),
            'max_vix_date': d['vix'].idxmax(), 'last_vix': float(d['vix'].iloc[-1])}

## Run

In [ ]:
print(fig_long_term())
print(fig_vix())

![sfm_ch8_long_term](./sfm_ch8_long_term.png)

Output: `sfm_ch8_long_term.pdf`

![sfm_ch8_vix](./sfm_ch8_vix.png)

Output: `sfm_ch8_vix.pdf`